# Quickstart: Model Visualization

This notebook loads a trained checkpoint, picks a random dataset sample, and compares **image vs ground truth vs prediction**.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import segmentation_models_pytorch as smp
import torch

In [ ]:
IMG_DIR = Path("../dataset/images")
MASK_DIR = Path("../dataset/masks")
CHECKPOINT_PATH = "../unet_best_v4.pth"
NUM_CLASSES = 4

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

mean = np.array(checkpoint["mean"], dtype=np.float32)
std = np.array(checkpoint["std"], dtype=np.float32)

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=NUM_CLASSES,
)
model.load_state_dict(checkpoint["model_state"])
model.to(device).eval()

print(f"Loaded model on: {device}")

In [ ]:
img_files = sorted(IMG_DIR.glob("*.npy"))
if not img_files:
    raise FileNotFoundError("No image tiles found in dataset/images")

img_path = random.choice(img_files)
idx = img_path.stem.split("_")[1]
mask_path = MASK_DIR / f"mask_{idx}.npy"

img = np.load(img_path).astype(np.float32)
mask = np.load(mask_path).astype(np.uint8)

if img.shape[0] == 3:
    img = np.transpose(img, (1, 2, 0))

img_norm = (img - mean) / (std + 1e-6)
img_tensor = torch.from_numpy(np.transpose(img_norm, (2, 0, 1))).unsqueeze(0).float().to(device)

with torch.no_grad():
    pred = torch.argmax(model(img_tensor), dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

cmap = np.array([
    [0, 0, 0],
    [0, 0, 255],
    [255, 255, 0],
    [255, 0, 0],
], dtype=np.uint8)

img_show = (img * 255).clip(0, 255).astype(np.uint8)
gt_rgb = cmap[mask]
pred_rgb = cmap[pred]

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(img_show)
plt.title("Satellite Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(gt_rgb)
plt.title("Ground Truth")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(pred_rgb)
plt.title("Prediction")
plt.axis("off")

plt.tight_layout()
plt.show()